# AmsiNet - Step 2: flat baselines on BSTD (word-level script identification)

**Runtime:** Colab Pro with a GPU (A100 if available, otherwise T4). Run Step 1 first; it creates `bstd_index.json` on your Drive.

What this notebook does
1. Cuts every labelled word out of the BSTD images once (letterboxed to 224x224) and stores the crops on the local disk.
2. Fine-tunes flat (single-agent, non-hierarchical) classifiers over the 10 scripts, one run per seed, with validation-based model selection and a single test evaluation.
3. Reports accuracy and macro-F1 as mean and 95% confidence interval over seeds, plus per-script F1 and confusion matrices.
4. Measures parameters, FLOPs and latency of the backbones you plan to use as agents (this fills Table 2 of the manuscript with measured values).

Protocol notes (write these in Section 4 of the paper)
- Splits are by **image** (train / val / test), so words from one photo never appear in two splits.
- The BSTD authors' published 12-way numbers use a different, class-balanced protocol on 12 languages. Do not compare with them directly unless you also run their `script_identification.zip` protocol.
- No horizontal flips: mirroring changes glyph shapes and therefore the script.

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')
!pip -q install timm gdown
import torch, timm
print('torch', torch.__version__, '| timm', timm.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (switch runtime to GPU)')

In [ ]:
# 2. Configuration and the Step-1 index
import json, os, glob, random, time, math
import numpy as np, pandas as pd

ROOT  = '/content/drive/MyDrive/amsinet'
LOCAL = '/content/bstd'

CFG = dict(
    MODELS = ['vit_base_patch16_224.augreg_in21k'],   # add e.g. 'convnext_tiny.fb_in22k' to get flat baselines with the agent backbones
    SEEDS  = [0],                                     # use [0, 1, 2, 3, 4] for the final paper numbers
    EPOCHS = 5, BATCH = 64, LR = 5e-5, WD = 0.05, IMG = 224, WORKERS = 8,
    LABEL_SMOOTHING = 0.1, CLASS_BALANCED_LOSS = False,
    APPLY_EXIF = True,                                # honour photo orientation tags (cv2 and browsers do)
    CROP_PAD = 0.04,                                  # context margin around each word box
)

with open(f'{ROOT}/data/bstd_index.json', 'r', encoding='utf-8') as f:
    D = json.load(f)
IMG_ROOT, SCRIPTS, INDEX = D['img_root'], D['scripts'], D['index']
LABEL = {s: i for i, s in enumerate(SCRIPTS)}
print(len(INDEX), 'images |', len(SCRIPTS), 'scripts:', SCRIPTS)

## 2b. Restore data in a fresh runtime
Colab gives each notebook its own temporary disk, so the images from Step 1 are not here. This cell (a) restores the finished crops from a tar file on your Drive if you saved one earlier, otherwise (b) downloads and unzips `detection.zip` again (about 17 GB). After the crops are extracted, the next cells save them to Drive as one tar file so later sessions skip the big download.

In [ ]:
import gdown, subprocess
CROPS = f"{LOCAL}/crops{CFG['IMG']}"
CROPS_TAR = f"{ROOT}/data/crops{CFG['IMG']}.tar"
os.makedirs(LOCAL, exist_ok=True)

if not os.path.isdir(CROPS) and os.path.exists(CROPS_TAR):
    print('restoring crops from Drive ...')
    subprocess.run(['tar', '-xf', CROPS_TAR, '-C', LOCAL], check=True)

if not os.path.isdir(CROPS) and not os.path.isdir(IMG_ROOT):
    print('downloading detection.zip (about 17 GB) ...')
    DET_ZIP = f'{LOCAL}/detection.zip'
    if not os.path.exists(DET_ZIP):
        gdown.download(id='1S7KUYfB-lQvbu6GtvZxxDPOD5ZZ080M0', output=DET_ZIP, quiet=False)
    subprocess.run(['unzip', '-q', '-o', DET_ZIP, '-d', LOCAL], check=True)
    IMG_ROOT = os.path.dirname(glob.glob(f'{LOCAL}/**/BSTD_v17.57.json', recursive=True)[0])
print('crops present:', os.path.isdir(CROPS), '| images present:', os.path.isdir(IMG_ROOT), '| IMG_ROOT =', IMG_ROOT)

## 3. Cut out the word crops (one-time, about 10-20 minutes; resumable)

In [ ]:
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None   # BSTD images come from the dataset authors; a few are huge panoramas
from concurrent.futures import ThreadPoolExecutor

CROPS = f"{LOCAL}/crops{CFG['IMG']}"
FILL = (124, 116, 104)

def letterbox(im, size):
    w, h = im.size
    s = size / max(w, h)
    nw, nh = max(1, int(round(w * s))), max(1, int(round(h * s)))
    im = im.resize((nw, nh), Image.BICUBIC)
    canvas = Image.new('RGB', (size, size), FILL)
    canvas.paste(im, ((size - nw) // 2, (size - nh) // 2))
    return canvas

def extract_one(rec):
    rows, im = [], None
    for k, w in enumerate(rec['words']):
        if not w['script']:
            continue
        d = f"{CROPS}/{rec['split']}/{w['script']}"
        os.makedirs(d, exist_ok=True)
        p = f"{d}/{rec['id']}_{k}.jpg"
        if not os.path.exists(p):
            if im is None:
                im = Image.open(os.path.join(IMG_ROOT, rec['image']))
                if CFG['APPLY_EXIF']:
                    im = ImageOps.exif_transpose(im)
                im = im.convert('RGB')
            x1, y1, x2, y2 = w['bbox']
            if (x2 - x1) < 4 or (y2 - y1) < 4:                       # ignore degenerate boxes
                continue
            px, py = (x2 - x1) * CFG['CROP_PAD'], (y2 - y1) * CFG['CROP_PAD']
            box = (max(0, int(x1 - px)), max(0, int(y1 - py)), min(im.width, int(math.ceil(x2 + px))), min(im.height, int(math.ceil(y2 + py))))
            if box[2] - box[0] < 2 or box[3] - box[1] < 2:
                continue
            letterbox(im.crop(box), CFG['IMG']).save(p, quality=95)
        rows.append((p, w['script'], w['lang'], rec['split'], rec['id'], k))
    return rows

def extract_all(index, workers=8):
    out = []
    with ThreadPoolExecutor(workers) as ex:
        for i, rows in enumerate(ex.map(extract_one, index)):
            out.extend(rows)
            if (i + 1) % 500 == 0:
                print(f'{i + 1}/{len(index)} images done, {len(out)} crops')
    return pd.DataFrame(out, columns=['path', 'script', 'lang', 'split', 'image_id', 'word_idx'])

In [ ]:
META = extract_all(INDEX)
META.to_csv(f'{ROOT}/data/crops_meta.csv', index=False)
print(META.groupby('split').size())
print(META.groupby(['split', 'script']).size().unstack(0).fillna(0).astype(int))

In [ ]:
# Save the crops to Drive as ONE file (fast to restore in any later session)
if not os.path.exists(CROPS_TAR):
    subprocess.run(['tar', '-cf', CROPS_TAR, '-C', LOCAL, f"crops{CFG['IMG']}"], check=True)
print('tar on Drive:', os.path.exists(CROPS_TAR), round(os.path.getsize(CROPS_TAR) / 1e9, 2), 'GB')

### Visual check (do this before training): the crops should show single words. If many look like background or rotated nonsense, set `APPLY_EXIF` to the other value, delete the crops folder and re-extract.

In [ ]:
import matplotlib.pyplot as plt
sm = META.sample(24, random_state=0)
fig, axs = plt.subplots(4, 6, figsize=(12, 8))
for ax, (_, r) in zip(axs.ravel(), sm.iterrows()):
    ax.imshow(Image.open(r.path)); ax.set_title(r.script[:12], fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Evaluation helpers (no GPU code here)

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix
from scipy import stats

def metrics(P, Y):
    pred = P.argmax(1)
    labs = list(range(len(SCRIPTS)))
    return {'acc': float((pred == Y).mean()),
            'macro_f1': float(f1_score(Y, pred, average='macro', labels=labs, zero_division=0)),
            'per_class_f1': f1_score(Y, pred, average=None, labels=labs, zero_division=0).tolist(),
            'confusion': confusion_matrix(Y, pred, labels=labs).tolist()}

def mean_ci(values, level=0.95):
    v = np.asarray(values, dtype=float)
    if len(v) < 2:
        return float(v.mean()), float('nan')
    half = stats.t.ppf(0.5 + level / 2, len(v) - 1) * v.std(ddof=1) / math.sqrt(len(v))
    return float(v.mean()), float(half)

def aggregate(results_dir, model_short):
    files = sorted(glob.glob(f'{results_dir}/{model_short}_seed*.json'))
    R = [json.load(open(f)) for f in files]
    if not R:
        return None
    row = {'model': model_short, 'seeds': len(R)}
    for split in ('val', 'test'):
        for m in ('acc', 'macro_f1'):
            mu, ci = mean_ci([r[split][m] for r in R])
            row[f'{split}_{m}'] = f'{100 * mu:.2f} +/- {100 * ci:.2f}' if ci == ci else f'{100 * mu:.2f}'
    pc = np.array([r['test']['per_class_f1'] for r in R])
    return row, pd.Series(100 * pc.mean(0), index=SCRIPTS, name=model_short)

## 5. Training (flat baseline: one classifier for all words)

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

DEV = 'cuda'
os.makedirs(f'{ROOT}/results', exist_ok=True); os.makedirs(f'{ROOT}/ckpt', exist_ok=True)

class Crops(Dataset):
    def __init__(self, df, tf):
        self.paths = df['path'].tolist(); self.labels = [LABEL[s] for s in df['script']]; self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert('RGB')), self.labels[i]

def make_tfs(model):
    cfg = timm.data.resolve_data_config({}, model=model)
    norm = T.Normalize(cfg['mean'], cfg['std'])
    train = T.Compose([T.RandomAffine(5, translate=(0.03, 0.03), scale=(0.9, 1.1), fill=FILL),
                       T.ColorJitter(0.3, 0.3, 0.2), T.ToTensor(), norm])      # no flips
    test = T.Compose([T.ToTensor(), norm])
    return train, test

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        with torch.autocast('cuda', dtype=torch.float16):
            lo = model(x.to(DEV, non_blocking=True))
        P.append(lo.float().softmax(1).cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def run(model_name, seed):
    short = model_name.split('.')[0]
    out = f'{ROOT}/results/{short}_seed{seed}.json'
    if os.path.exists(out):
        print('skip (done):', out); return
    set_seed(seed)
    model = timm.create_model(model_name, pretrained=True, num_classes=len(SCRIPTS)).to(DEV)
    ttf, etf = make_tfs(model)
    tr, va, te = (META[META.split == s] for s in ('train', 'val', 'test'))
    dl = lambda df, tf, sh: DataLoader(Crops(df, tf), batch_size=CFG['BATCH'], shuffle=sh, num_workers=CFG['WORKERS'], pin_memory=True, drop_last=sh)
    L_tr, L_va, L_te = dl(tr, ttf, True), dl(va, etf, False), dl(te, etf, False)
    w = None
    if CFG['CLASS_BALANCED_LOSS']:
        cnt = tr['script'].value_counts().reindex(SCRIPTS).values.astype(float)
        w = torch.tensor((1 / np.sqrt(cnt)) / (1 / np.sqrt(cnt)).mean(), dtype=torch.float32, device=DEV)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG['LABEL_SMOOTHING'])
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['LR'], weight_decay=CFG['WD'])
    total = CFG['EPOCHS'] * len(L_tr); warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler('cuda')
    best, best_state, hist = -1, None, []
    for ep in range(CFG['EPOCHS']):
        model.train(); t0 = time.time(); run_loss = 0
        for x, y in L_tr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            with torch.autocast('cuda', dtype=torch.float16):
                loss = crit(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run_loss += loss.item()
        mv = metrics(*predict(model, L_va))
        hist.append({'epoch': ep + 1, 'loss': run_loss / len(L_tr), 'val_acc': mv['acc'], 'val_macro_f1': mv['macro_f1']})
        print(f"{short} seed{seed} ep{ep + 1}: loss {run_loss / len(L_tr):.4f} val acc {mv['acc']:.4f} macro-F1 {mv['macro_f1']:.4f} ({time.time() - t0:.0f}s)")
        if mv['macro_f1'] > best:
            best = mv['macro_f1']; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}; best_val = mv
    model.load_state_dict(best_state)
    mt = metrics(*predict(model, L_te))                      # test is touched exactly once, with the selected model
    torch.save(best_state, f'{ROOT}/ckpt/{short}_seed{seed}.pt')
    json.dump({'model': model_name, 'seed': seed, 'cfg': {k: v for k, v in CFG.items()}, 'history': hist,
               'val': best_val, 'test': mt}, open(out, 'w'))
    print(f"TEST {short} seed{seed}: acc {mt['acc']:.4f} macro-F1 {mt['macro_f1']:.4f}")

In [ ]:
for m in CFG['MODELS']:
    for s in CFG['SEEDS']:
        run(m, s)

## 6. Results table (mean and 95% confidence interval over seeds)

In [ ]:
rows, pcs = [], []
for m in CFG['MODELS']:
    a = aggregate(f'{ROOT}/results', m.split('.')[0])
    if a: rows.append(a[0]); pcs.append(a[1])
tab = pd.DataFrame(rows); display(tab)
display(pd.DataFrame(pcs).round(1))
tab.to_csv(f'{ROOT}/results/baseline_table.csv', index=False)

## 7. Parameters, FLOPs and latency of the agent backbones (measured, for Table 2)
FLOPs here count 2 x multiply-accumulates for one 224x224 image (state this convention in the paper). Latency is the median of 200 runs at batch size 1 in fp32 on the GPU shown above. If timm reports an unknown model name, list valid ones with `timm.list_models('*convnext_tiny*', pretrained=True)` and substitute.

In [ ]:
from torch.utils.flop_counter import FlopCounterMode

BACKBONES = {'Page agent': 'mobilevit_xs.cvnets_in1k', 'Block agent': 'convnext_tiny.fb_in22k',
             'Line agent': 'vit_small_patch16_224.augreg_in21k', 'Word agent': 'efficientvit_b0.r224_in1k',
             'Flat baseline': 'vit_base_patch16_224.augreg_in21k'}

def efficiency(name, size=224):
    m = timm.create_model(name, pretrained=False, num_classes=len(SCRIPTS)).to(DEV).eval()
    x = torch.randn(1, 3, size, size, device=DEV)
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        m(x)
    gflops = fc.get_total_flops() / 1e9
    params = sum(p.numel() for p in m.parameters()) / 1e6
    with torch.no_grad():
        for _ in range(50): m(x)
        torch.cuda.synchronize(); ts = []
        for _ in range(200):
            t = time.perf_counter(); m(x); torch.cuda.synchronize(); ts.append((time.perf_counter() - t) * 1000)
    return round(params, 2), round(gflops, 2), round(float(np.median(ts)), 2)

eff = pd.DataFrame([(k, v) + efficiency(v) for k, v in BACKBONES.items()], columns=['agent', 'timm model', 'params (M)', 'GFLOPs', 'latency (ms)'])
display(eff); eff.to_csv(f'{ROOT}/results/backbone_efficiency.csv', index=False)
print('GPU:', torch.cuda.get_device_name(0))